##__Data-Driven Product Optimization Using A/B Testing & User Engagement Analytics__
---

#### __Data Loading & Setup__
---

##### Import the required libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
!pip install opendatasets
import opendatasets as od
pd.set_option('display.max_columns', None)

##### Now lets download the dataset from kaggle using `opendatasets` library

In [ ]:
od.download("https://www.kaggle.com/datasets/sanxhi/ab-testing-data-simulated-web-user-engagement")

Skipping, found downloaded files in "./ab-testing-data-simulated-web-user-engagement" (use force=True to force download)


##### Lets load the downloaded dataset into a pandas DataFrame, parsing the `click_time` column as datetime objects. And display some rows

In [ ]:
file=pd.read_csv(r'./ab-testing-data-simulated-web-user-engagement/ab_test_dataset.csv', parse_dates=['click_time'], date_format='%Y-%m-%d %H:%M:%S')
file.head()

,click,group,session_time,click_time,device_type,referral_source
0,1,exp,0.040363,NaT,mobile,search
1,0,exp,1.639570,2024-01-14 22:15:00,mobile,email
2,0,exp,2.961715,2024-01-01 15:36:00,mobile,direct
3,1,exp,2.784540,2024-01-04 17:39:00,desktop,email
4,0,exp,2.495874,2024-01-07 17:31:00,mobile,social


#### __Data Cleaning & Preprocessing__
---

##### Now, lets look into the concise summary of the DataFrame, including data types and non-null values for each column

In [ ]:
file.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200020 entries, 0 to 200019
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   click            200020 non-null  int64         
 1   group            198050 non-null  object        
 2   session_time     200020 non-null  float64       
 3   click_time       198019 non-null  datetime64[ns]
 4   device_type      200020 non-null  object        
 5   referral_source  199031 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(3)
memory usage: 9.2+ MB


##### We can see there are some null values in `group`, `click_time` and `referral_source` column, so we will remove some of these null rows but before that lets check for duplicates

In [ ]:
print(file.duplicated().sum())

31


##### Now, lets addresses missing values by dropping rows where the `group` column is null, removes duplicate rows, and fills missing `referral_source` values with `unknown`. And then print the updated DataFrame info

In [ ]:
file.dropna(subset=['group'],inplace=True)
file = file.drop_duplicates().reset_index(drop=True)
file['referral_source'] = file['referral_source'].fillna('unknown')
file.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198019 entries, 0 to 198018
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   click            198019 non-null  int64         
 1   group            198019 non-null  object        
 2   session_time     198019 non-null  float64       
 3   click_time       196048 non-null  datetime64[ns]
 4   device_type      198019 non-null  object        
 5   referral_source  198019 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(3)
memory usage: 9.1+ MB


*we dont need to remove or modify null values in `click_time` column for now as this column is not necessary for our immediate goal to perform Z-test, but we will do check the distribution along with `click` column to understand if most of the null values are in either variables of click column*

In [ ]:
file[file['click_time'].isna()]['click'].value_counts()

,count
click,
0,1300
1,671


##### So, lets re-checks for missing values and duplicate rows after cleaning the data

In [ ]:
print(file.isna().sum())
print(file.duplicated().sum())

click                 0
group                 0
session_time          0
click_time         1971
device_type           0
referral_source       0
dtype: int64
0


##### Now lets check data distributions among each of the columns to inspect unique values and potential inconsistencies

In [ ]:
for i in file.columns:
  print(file[i].value_counts())

click
0    128947
1     69072
Name: count, dtype: int64
group
exp     97052
con     97002
con      1000
Exp       995
A         992
a         978
Name: count, dtype: int64
session_time
807.785808    1970
0.541924         1
3.453617         1
3.441587         1
2.780007         1
              ... 
9.785725         1
4.801385         1
2.241849         1
6.971498         1
9.187490         1
Name: count, Length: 196050, dtype: int64
click_time
2024-01-05 01:43:00    25
2024-01-08 07:26:00    25
2024-01-13 16:47:00    24
2024-01-05 08:17:00    24
2024-01-10 03:46:00    24
                       ..
2024-01-03 17:58:00     1
2024-01-07 09:59:00     1
2024-01-01 08:17:00     1
2024-01-06 08:49:00     1
2024-01-05 22:03:00     1
Name: count, Length: 20157, dtype: int64
device_type
mobile     137222
desktop     58826
MOBILE        988
Desktop       983
Name: count, dtype: int64
referral_source
search      58342
email       49129
social      38652
direct      29420
ads         19516
seach     

##### We can observe some inconsistencies like in `group`, `device_type` and `referral_source` column. So, now lets standardize the variables of those columns

##### So, let's defines a function correction to standardize string columns by stripping whitespace and converting text to lowercase

In [ ]:
def correction(df):
  for i in df.select_dtypes(include='O').columns:
    df[i]=df[i].str.strip().str.lower()
  return df

*we need to check `a` in `group` column before acting*

*Exploratory analysis revealed a third group label (‘A’) with engagement metrics distinct from both control and experimental groups. Since its treatment definition was ambiguous, these rows were excluded to preserve the validity of the A/B comparison*

*Initial exploration revealed inconsistent experiment labels (con, exp, a) with materially different click-through rates. After standardizing labels, the a group exhibited intermediate behavior inconsistent with either control or treatment, suggesting mixed or ambiguous exposure. To preserve experimental validity, analysis was restricted to clearly defined control (con) and experimental (exp) groups*

*The raw group column contains inconsistent and ambiguous labels; after analysis, only con and exp represent stable experimental conditions, and the A/a group should be excluded from the A/B test*

*Click timestamps contained missing values due to logging gaps. Since the primary outcome variable (click) was fully observed, missing timestamps were retained without imputation to avoid introducing artificial temporal bias. Click_time was excluded from hypothesis testing*

##### Now, let's apply the correction function to the DataFrame, standardizes the `referral_source` column by replacing 'seach' with 'search', and removes rows where `group` is 'a' (likely an erroneous entry).

In [ ]:
file1=correction(file)
file1['referral_source']=file1['referral_source'].str.replace('seach', 'search')
file1=file1.drop(file1[file1['group']=='a'].index)
file1.reset_index(drop=True,inplace=True)
file1.head()

,click,group,session_time,click_time,device_type,referral_source
0,1,exp,0.040363,NaT,mobile,search
1,0,exp,1.639570,2024-01-14 22:15:00,mobile,email
2,0,exp,2.961715,2024-01-01 15:36:00,mobile,direct
3,1,exp,2.784540,2024-01-04 17:39:00,desktop,email
4,0,exp,2.495874,2024-01-07 17:31:00,mobile,social


##### Now, lets re-checks the value counts for all columns after standardization

In [ ]:
for i in file1.columns:
  print(file1[i].value_counts())

click
0    127672
1     68377
Name: count, dtype: int64
group
exp    98047
con    98002
Name: count, dtype: int64
session_time
807.785808    1951
3.269404         1
3.949001         1
17.472070        1
1.368228         1
              ... 
5.256722         1
7.222205         1
3.735825         1
1.419758         1
8.455060         1
Name: count, Length: 194099, dtype: int64
click_time
2024-01-05 01:43:00    25
2024-01-05 08:17:00    24
2024-01-08 07:26:00    24
2024-01-13 16:47:00    24
2024-01-10 03:46:00    23
                       ..
2024-01-01 08:17:00     1
2024-01-04 13:05:00     1
2024-01-13 07:56:00     1
2024-01-03 17:58:00     1
2024-01-01 11:57:00     1
Name: count, Length: 20157, dtype: int64
device_type
mobile     136836
desktop     59213
Name: count, dtype: int64
referral_source
search     58771
email      48648
social     39223
direct     29088
ads        19347
unknown      972
Name: count, dtype: int64


##### Lets re-checks for missing values again

In [ ]:
file1.isna().sum()

,0
click,0
group,0
session_time,0
click_time,1953
device_type,0
referral_source,0


#### **Experiment Context & Goal**
---

##### Now, before we move to testing, lets look at some points to validate our experiment later. So, we will look into some points like:
* Sample Ratio Mismatch (SRM) check
* Define hypotheses
* Compute the key metrics
* Confidence Interval (CI)

##### Now, we will create a copy of our original dataset to avoid modifying it

In [ ]:
file2=file1.copy()

##### ***Sample Ratio Mismatch (SRM) check***

##### Lets check the proportional distribution of the `group` column to check for balanced groups

In [ ]:
file2['group'].value_counts(normalize=True)

,proportion
group,
exp,0.500115
con,0.499885


*Traffic is evenly split between control and experiment groups (~50/50), indicating proper randomization.*

##### ***Define hypotheses***

Null hypothesis (H₀)

Conversion rate of control = conversion rate of experiment

Alternative hypothesis (H₁)

Conversion rate of control ≠ conversion rate of experiment

##### ***Compute the key metrics***

##### Now, lets calculates the total users, total clicks, and overall conversion rate for each group (control and experimental)

In [ ]:
summary=file2.groupby('group')['click'].agg(users='count', clicks='sum', conversion_rate='mean')
summary

,users,clicks,conversion_rate
group,,,
con,98002,19722,0.201241
exp,98047,48655,0.496242


##### Now, lets calculates the absolute and relative lift in conversion rates between the experimental(exp) and control(con) groups based on the summary statistics to understand the magnitude of change beween control(con) and experiment(exp) groups

In [ ]:
abs_lift = summary.loc['exp', 'conversion_rate'] - summary.loc['con', 'conversion_rate']
rltv_lift = abs_lift / summary.loc['con', 'conversion_rate']
print(f"Absolute Lift: {abs_lift}")
print(f"Relative Lift: {rltv_lift}")

Absolute Lift: 0.2950008074128205
Relative Lift: 1.4659095998413565


##### __*Confidence Interval (CI)*__

##### Lets define a function `cnfd_intv` to calculate the effect size and confidence interval (CI) for the difference in conversion rates

In [ ]:
import math

def cnfd_intv(df, column2, column3):
  se=math.sqrt((df.loc['exp', column3] * (1-df.loc['exp', column3])/
                df.loc['exp', column2]) + (df.loc['con', column3] * (1-df.loc['con', column3])/df.loc['con', column2]))
  z=1.96
  diff=df.loc['exp', column3]-df.loc['con', column3]
  ci_low = diff - z * se
  ci_high = diff + z * se
  return diff, ci_low, ci_high

##### Now lets call the `cnfd_intv` function to calculate the `effect size` and `confidence interval` for the overall A/B test and prints the results

In [ ]:
effect_size, cnfd_intv_low, cnfd_intv_high=cnfd_intv(summary, 'clicks', 'conversion_rate')
print(f"Effect Size: {effect_size}")
print(f"95% Confidence Interval: ({cnfd_intv_low}, {cnfd_intv_high})")

Effect Size: 0.2950008074128205
95% Confidence Interval: (0.28785597814808084, 0.3021456366775601)


#### __*Z_test*__

##### A two-sample Z-test for proportions was chosen to compare click-through rates between the control and experimental groups. This test is appropriate because the outcome variable is binary, the samples are independent, and both groups have large sample sizes. A two-tailed test was conducted at a 5% significance level.

##### Now, lets define a function `ztest` to perform a two-sample Z-test for proportions, which helps to determine the statistical significance of the observed difference in conversion rates

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

def ztest(df, column1, column2):
  counts=df[column1].values
  nobs=df[column2].values
  z_stat, p_value = proportions_ztest(counts, nobs, alternative='two-sided')
  return z_stat, p_value

##### Now, lets call the `ztest` function to calculate the Z-statistic and P-value for the overall A/B test and prints the results

In [ ]:
zstat, pvalue=ztest(summary, 'clicks', 'users')
print(f"Z-statistic: {zstat}")
print(f"P-value: {pvalue}")

Z-statistic: -137.0368264593863
P-value: 0.0


*Hypothesis Test Result: A two-sample Z-test for proportions yielded a test statistic of |Z| = 137 with a p-value effectively equal to zero, providing overwhelming evidence against the null hypothesis of equal CTRs.*

*Note: The magnitude of the observed effect and Z-statistic is unusually large for real-world A/B tests and is expected given the synthetic nature of the dataset.*

##### __Robustness, Segmentation & Sensitivity Analysis__
---

##### **Guardrail Metrics (Engagement)**

This cell calculates the mean session time for both the control and experimental groups.

In [ ]:
file2.groupby('group')['session_time'].mean()

,session_time
group,
con,13.292745
exp,12.672549


This cell calculates the median session time for both the control and experimental groups.

In [ ]:
file2.groupby('group')['session_time'].median()

,session_time
group,
con,3.522824
exp,3.520125


*Metric Sensitivity Analysis: To ensure increased click-through did not come at the expense of engagement quality, session duration was analyzed as a secondary metric. While mean session time showed a slight decrease in the experimental group, median session time remained effectively unchanged, indicating that typical user engagement was stable. This suggests that the observed CTR uplift reflects genuine engagement rather than superficial clicks.*

*An A/B experiment was conducted to evaluate the impact of the experimental variant on user click-through rate (CTR). The experiment resulted in a statistically significant and practically meaningful increase in CTR (~29.5 percentage points, ~1.46× relative lift). The effect was consistent across device types, time-of-day segments, and referral sources. Sensitivity analyses confirmed robustness to sample size variation and metric definition.*

##### **Heterogeneous Treatment Effects (HTE)**

##### Let's defines a utility function lift to calculate both absolute and relative lift, which will be reused for segmented analyses

In [ ]:
def lift(df, grp1, grp2):
  df['abs_lift'] = df[grp2]-df[grp1]
  df['rltv_lift'] = df['abs_lift']/df[grp1]
  return df

##### Now, let's calculate the Click-Through Rate (CTR) for each group, segmented by device_type, and then uses the lift function to compute absolute and relative lift for each device category

In [ ]:
ctr_by_device=file2.groupby(['group', 'device_type'])['click'].mean().unstack().T
ctr_by_device=lift(ctr_by_device, 'con', 'exp')
ctr_by_device

group,con,exp,abs_lift,rltv_lift
device_type,,,,
desktop,0.203150,0.499663,0.296513,1.459579
mobile,0.200418,0.494756,0.294338,1.468623


##### Now, let's calculate the Click-Through Rate (CTR) for each group, segmented by referral_source, and then uses the lift function to compute absolute and relative lift for each referral source

In [ ]:
ctr_by_source=file2.groupby(['group', 'referral_source'])['click'].mean().unstack().T
ctr_by_source=lift(ctr_by_source, 'con', 'exp')
ctr_by_source

group,con,exp,abs_lift,rltv_lift
referral_source,,,,
ads,0.202261,0.496413,0.294152,1.454316
direct,0.203158,0.495608,0.292450,1.439520
email,0.197503,0.495523,0.298021,1.508944
search,0.201956,0.495208,0.293252,1.452062
social,0.203220,0.498555,0.295335,1.453281
unknown,0.186747,0.516878,0.330131,1.767796


*Sessions with missing click timestamps were retained for all primary A/B analyses, as missing timestamps primarily correspond to non-click events and do not affect the unit of randomization. For time-of-day robustness analysis, sessions without timestamps were excluded, as temporal segmentation requires observed interaction time.*

##### **Temporal Robustness (Time-based Checks)**

Now, let's filters the DataFrame to include only sessions where click_time is not null, which is necessary for time-based analyses. It then displays the first 5 rows of this new DataFrame

In [ ]:
file_time=file2.dropna(subset=['click_time']).reset_index(drop=True)
file_time.head()

,click,group,session_time,click_time,device_type,referral_source
0,0,exp,1.639570,2024-01-14 22:15:00,mobile,email
1,0,exp,2.961715,2024-01-01 15:36:00,mobile,direct
2,1,exp,2.784540,2024-01-04 17:39:00,desktop,email
3,0,exp,2.495874,2024-01-07 17:31:00,mobile,social
4,0,exp,10.405808,2024-01-05 11:02:00,desktop,social


This cell displays a concise summary of the time-filtered DataFrame, including data types and non-null values.

In [ ]:
file_time.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194096 entries, 0 to 194095
Data columns (total 6 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   click            194096 non-null  int64         
 1   group            194096 non-null  object        
 2   session_time     194096 non-null  float64       
 3   click_time       194096 non-null  datetime64[ns]
 4   device_type      194096 non-null  object        
 5   referral_source  194096 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(3)
memory usage: 8.9+ MB


This cell extracts the hour from the click_time column and stores it in a new column named time. It then displays the first 5 rows.

In [ ]:
file_time['time']=file_time['click_time'].apply(lambda x: x.hour)
file_time.head()

,click,group,session_time,click_time,device_type,referral_source,time
0,0,exp,1.639570,2024-01-14 22:15:00,mobile,email,22
1,0,exp,2.961715,2024-01-01 15:36:00,mobile,direct,15
2,1,exp,2.784540,2024-01-04 17:39:00,desktop,email,17
3,0,exp,2.495874,2024-01-07 17:31:00,mobile,social,17
4,0,exp,10.405808,2024-01-05 11:02:00,desktop,social,11


This cell creates categorical 'time_bucket' labels (Night, Morning, Afternoon, Evening) based on the hour extracted from click_time.

In [ ]:
bins = [-1, 5, 11, 17, 23]
labels = ['Night', 'Morning', 'Afternoon', 'Evening']
file_time['time_bucket'] = pd.cut(file_time['time'], bins=bins, labels=labels)
file_time.head()

,click,group,session_time,click_time,device_type,referral_source,time,time_bucket
0,0,exp,1.639570,2024-01-14 22:15:00,mobile,email,22,Evening
1,0,exp,2.961715,2024-01-01 15:36:00,mobile,direct,15,Afternoon
2,1,exp,2.784540,2024-01-04 17:39:00,desktop,email,17,Afternoon
3,0,exp,2.495874,2024-01-07 17:31:00,mobile,social,17,Afternoon
4,0,exp,10.405808,2024-01-05 11:02:00,desktop,social,11,Morning


This cell displays the proportional distribution of the newly created time_bucket column.

In [ ]:
file_time['time_bucket'].value_counts(normalize=True)

,proportion
time_bucket,
Afternoon,0.251489
Morning,0.249995
Evening,0.249624
Night,0.248892


This cell calculates the Click-Through Rate (CTR) for each group, segmented by time_bucket.

In [ ]:
ctr_by_time=file_time.groupby(['group', 'time_bucket'])['click'].mean().reset_index()
ctr_by_time

,group,time_bucket,click
0,con,Night,0.198997
1,con,Morning,0.200860
2,con,Afternoon,0.203394
3,con,Evening,0.201623
4,exp,Night,0.497870
5,exp,Morning,0.496487
6,exp,Afternoon,0.498745
7,exp,Evening,0.492055


This cell reshapes the ctr_by_time data into a pivot table to compare 'con' and 'exp' groups across different time buckets, and then calculates the absolute and relative lift for each time segment.

In [ ]:
ctr_time_pivot = ctr_by_time.pivot(index='time_bucket', columns='group', values='click')
ctr_time_pivot=lift(ctr_time_pivot, 'con', 'exp')
#ctr_time_pivot['absolute_lift'] = (ctr_time_pivot['exp'] - ctr_time_pivot['con'])
#ctr_time_pivot['relative_lift'] = ctr_time_pivot['absolute_lift'] / ctr_time_pivot['con']
ctr_time_pivot

group,con,exp,abs_lift,rltv_lift
time_bucket,,,,
Night,0.198997,0.497870,0.298873,1.501900
Morning,0.200860,0.496487,0.295627,1.471805
Afternoon,0.203394,0.498745,0.295351,1.452112
Evening,0.201623,0.492055,0.290432,1.440476


##### **Sensitivity Analysis**

This cell creates a downsampled version of the file2 DataFrame (file3) by randomly selecting 50% of the data from each group, which is used for sensitivity analysis. It then displays the DataFrame information.

In [ ]:
file3 =(file2.groupby('group', group_keys=False).apply(lambda x: x.sample(frac=0.5, random_state=42)))
file3.reset_index(drop=True, inplace=True)
file3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98025 entries, 0 to 98024
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   click            98025 non-null  int64         
 1   group            98025 non-null  object        
 2   session_time     98025 non-null  float64       
 3   click_time       97051 non-null  datetime64[ns]
 4   device_type      98025 non-null  object        
 5   referral_source  98025 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(3)
memory usage: 4.5+ MB


This cell displays the proportional distribution of the 'group' column in the downsampled DataFrame (file3) to ensure groups remain balanced.

In [ ]:
file3['group'].value_counts(normalize=True)

,proportion
group,
exp,0.500117
con,0.499883


This cell calculates the total users, total clicks, and overall conversion rate for each group within the downsampled DataFrame (file3).

In [ ]:
summary1=file3.groupby('group')['click'].agg(users='count', clicks='sum', conversion_rate='mean')
summary1

,users,clicks,conversion_rate
group,,,
con,49001,9891,0.201853
exp,49024,24341,0.496512


This cell calculates the absolute and relative lift in conversion rates for the downsampled data, comparing the experimental and control groups.

In [ ]:
summary1['abs_lift'] = summary1.loc['exp', 'conversion_rate'] - summary1.loc['con', 'conversion_rate']
summary1['rltv_lift'] = summary1['abs_lift'] / summary1.loc['con', 'conversion_rate']
summary1

,users,clicks,conversion_rate,abs_lift,rltv_lift
group,,,,,
con,49001,9891,0.201853,0.294659,1.45977
exp,49024,24341,0.496512,0.294659,1.45977


This cell performs a Z-test on the downsampled data to check the statistical significance of the observed difference in conversion rates for the sensitivity analysis.

In [ ]:
zstat_sensitivity, pvalue_sensitivity = ztest(summary1, 'clicks', 'users')
print(f"Z-statistic for sensitivity analysis: {zstat_sensitivity}")
print(f"P-value for sensitivity analysis: {pvalue_sensitivity}")

Z-statistic for sensitivity analysis: -96.75917709920745
P-value for sensitivity analysis: 0.0


### __Conclusion & Business Recommendation__
---

- The experimental variant shows a substantial improvement in click-through rate compared to the control group
- The observed lift is both practically meaningful and statistically significant
- The hypothesis test results strongly reject the null hypothesis
- Confidence intervals further support that the observed improvement is not due to random variation
- The performance uplift remains consistent across:
  -  Device types
  -  Referral sources
  -  Time-based segments
- This indicates that the treatment effect is robust and generalizable
- No significant degradation observed in engagement metrics such as session duration
- Median session behavior remains stable, suggesting no negative user experience impact


 __👉Recommendation__

 The experimental variant should be rolled out to all users, as it significantly improves engagement without introducing measurable risks.

__🚧Limitations__

The dataset is simulated, and real-world user behavior may introduce additional variability. External factors (seasonality, user intent shifts) are not fully captured.

###